# 04 — Return Horizon Sweep: Finding the Ceiling on Price Data

**The last free lever before any paid data step.** The plan was: exhaust free
experiments, and only pay for fundamentals if there is evidence a return signal is
*reachable*. So far:

- 5-day cross-sectional IC ~0.014 (thin, positive every fold)
- 20-day IC ~0.027 (roughly double — lengthening helped)
- Short interest: no lift at any horizon or in 2022 (the one non-price feature tested)

The open question this notebook answers: **does the return edge keep growing as the
horizon lengthens, or does it plateau?** Slower signals (value reversion, sustained
momentum, drift) live at monthly-to-quarterly horizons and are harder to arbitrage
instantly. If 60–120 day shows a materially stronger, more stable edge than 20-day,
that is the evidence that justifies buying point-in-time fundamentals (which are
themselves monthly/quarterly signals). If it stays thin and fragile, the honest read
is that returns have a low ceiling on price data, and the volatility engine is the
deliverable.

## Design

- **Horizons: 5, 20, 60, 120 trading days.** Target recomputed inline per horizon.
- **Cross-sectional rank framing** — the edge that exists is relative, not directional.
- **Ridge and XGBoost** — prior work showed returns are linear (Ridge >= XGBoost).
- **Metrics: rank IC and cost-aware net long-short spread**, with the purge gap and
  rebalance frequency scaled to each horizon.
- Walk-forward, purge, train-only scaling — unchanged.

This is decision-useful either way: it turns "should I pay for fundamentals?" into an
evidence-based call by mapping the realistic return ceiling on free price data.

In [ ]:
from pathlib import Path
import sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True; plt.rcParams["grid.alpha"] = 0.3
warnings.filterwarnings("ignore")

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
for parent in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (parent / 'src').is_dir():
        PROJECT_ROOT = parent; break
else:
    raise RuntimeError('Run from inside the StockForecastRisk repository.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.forecast_engine.features.schema import FEATURE_NAMES, NON_FEATURE_COLUMNS
from src.forecast_engine.data.loader import load_processed_features

data = load_processed_features()
data["date"] = pd.to_datetime(data["date"])
data = data.sort_values(["symbol", "date"]).reset_index(drop=True)
print(f'Loaded {len(data):,} rows, {data["symbol"].nunique()} tickers, '
      f'{data["date"].min().date()} -> {data["date"].max().date()}')

## Features — stationary price + macro, excluding raw levels and short interest

Short interest is excluded here: it added nothing (notebook 01) and only covers 2018+,
which would shorten the panel. This sweep isolates the horizon effect on price features.

In [ ]:
NON_STATIONARY_LEVELS = ["sma_10","sma_20","sma_50","sma_200","ema_12","ema_26","vwap_20"]
SI = ["short_interest","short_interest_change","days_to_cover","days_to_cover_change"]
EXCLUDE = set(NON_FEATURE_COLUMNS) | set(NON_STATIONARY_LEVELS) | set(SI) | {"short_history","history_rows"}
feature_cols = [c for c in FEATURE_NAMES if c not in EXCLUDE and c in data.columns and not data[c].isna().all()]
print(f"features: {len(feature_cols)}")

In [ ]:
HORIZONS = [5, 20, 60, 120]  # trading days

def forward_log_return(group, h):
    c = group["adj_close"].astype(float)
    return np.log(c.shift(-h) / c)

for h in HORIZONS:
    data[f"fwd_ret_{h}"] = data.groupby("symbol", group_keys=False).apply(
        lambda g: forward_log_return(g, h))
print("forward-return targets built for horizons:", HORIZONS)
print(data[[f"fwd_ret_{h}" for h in HORIZONS]].std().round(4).to_frame("target_std"))

## Shared harness — walk-forward, IC, cost-aware spread (scaled per horizon)

In [ ]:
N_SPLITS = 5
def walk_forward_splits(frame, purge_days):
    dates = np.sort(frame["date"].unique())
    edges = np.array_split(dates, N_SPLITS + 1)
    for f in range(N_SPLITS):
        cutoff = edges[f][-1] - pd.Timedelta(days=purge_days * 2)  # calendar cushion
        tr = frame.index[frame["date"] <= cutoff]
        te = frame.index[frame["date"].isin(edges[f + 1])]
        if len(tr) and len(te):
            yield tr, te

def daily_rank_ic(dates, y, p):
    df = pd.DataFrame({"d": dates, "y": y, "p": p})
    vals = []
    for _, g in df.groupby("d"):
        if g["y"].nunique() > 2 and g["p"].nunique() > 2:
            c = spearmanr(g["p"], g["y"]).correlation
            if np.isfinite(c): vals.append(c)
    return (np.mean(vals) if vals else np.nan)

DECILE, COST_BPS = 0.10, 5.0
def net_spread(dates, y, p):
    df = pd.DataFrame({"d": dates, "y": y, "p": p}); nets = []
    for _, g in df.groupby("d"):
        if len(g) < 20: continue
        hi = g["p"] >= g["p"].quantile(0.9); lo = g["p"] <= g["p"].quantile(0.1)
        if hi.sum() and lo.sum():
            nets.append((g.loc[hi,"y"].mean() - g.loc[lo,"y"].mean()) - 4*COST_BPS/1e4)
    return float(np.mean(nets)) if nets else np.nan

def make_reg():
    return XGBRegressor(n_estimators=300, max_depth=5, learning_rate=0.05, subsample=0.8,
        colsample_bytree=0.8, n_jobs=-1, random_state=42, verbosity=0, tree_method="hist")

## Run the sweep across horizons

For each horizon: rebuild the model frame (dropping rows whose forward return is NaN
at that horizon — longer horizons lose more tail rows), scale the purge gap to the
horizon, and evaluate Ridge and XGBoost. Annualise the net spread by the horizon's
rebalance frequency so the spreads are comparable across horizons.

In [ ]:
results = []
for h in HORIZONS:
    target = f"fwd_ret_{h}"
    md_h = data.dropna(subset=feature_cols + [target]).reset_index(drop=True)
    X = md_h[feature_cols].to_numpy(np.float32)
    y = md_h[target].to_numpy(np.float32)
    dts = md_h["date"].to_numpy()
    folds = list(walk_forward_splits(md_h, purge_days=h))
    rebal_per_year = 252 / h

    for i, (tr, te) in enumerate(folds, 1):
        sc = StandardScaler().fit(X[tr])
        rp = Ridge(alpha=1.0).fit(sc.transform(X[tr]), y[tr]).predict(sc.transform(X[te]))
        xp = make_reg().fit(X[tr], y[tr]).predict(X[te])
        for name, pred in [("ridge", rp), ("xgboost", xp)]:
            ns = net_spread(dts[te], y[te], pred)
            results.append({"horizon": h, "fold": i, "model": name,
                            "ic": daily_rank_ic(dts[te], y[te], pred),
                            "net": ns,
                            "net_annualised": ns * rebal_per_year if ns==ns else np.nan})
    print(f"horizon {h}d done ({len(md_h):,} rows, {len(folds)} folds)")

res = pd.DataFrame(results)
summary = res.groupby(["horizon","model"])[["ic","net","net_annualised"]].mean()
print("\nReturn edge by horizon (mean across folds):")
display(summary.round(5))

In [ ]:
# The headline: does IC grow with horizon, and does it stay positive across folds?
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

# (a) mean IC vs horizon
for model, color in [("ridge","tab:blue"),("xgboost","tab:orange")]:
    s = summary.xs(model, level="model")["ic"]
    axes[0].plot(s.index, s.values, "-o", color=color, label=model)
axes[0].axhline(0, color="grey", lw=0.8)
axes[0].set_title("Mean rank IC vs horizon"); axes[0].set_xlabel("horizon (days)")
axes[0].set_ylabel("mean IC"); axes[0].legend()

# (b) IC consistency: fraction of folds positive
posfrac = res.groupby(["horizon","model"])["ic"].apply(lambda s: (s>0).mean()).unstack("model")
posfrac.plot(marker="o", ax=axes[1])
axes[1].axhline(1.0, color="green", ls="--", alpha=0.5, label="all folds positive")
axes[1].set_title("Fraction of folds with positive IC"); axes[1].set_xlabel("horizon (days)")
axes[1].set_ylabel("fold positivity"); axes[1].legend()

# (c) annualised net spread vs horizon
for model, color in [("ridge","tab:blue"),("xgboost","tab:orange")]:
    s = summary.xs(model, level="model")["net_annualised"]
    axes[2].plot(s.index, s.values, "-o", color=color, label=model)
axes[2].axhline(0, color="red", ls="--")
axes[2].set_title("Annualised net long-short spread vs horizon")
axes[2].set_xlabel("horizon (days)"); axes[2].set_ylabel("annualised net spread"); axes[2].legend()

plt.tight_layout(); plt.show()

In [ ]:
# IC by fold at each horizon — is the longer-horizon edge stable or lumpy?
fig, axes = plt.subplots(1, len(HORIZONS), figsize=(4*len(HORIZONS), 3.5), sharey=True)
for ax, h in zip(axes, HORIZONS):
    for model, color in [("ridge","tab:blue"),("xgboost","tab:orange")]:
        sub = res[(res.horizon==h)&(res.model==model)].sort_values("fold")
        ax.plot(sub.fold, sub.ic, "-o", color=color, label=model)
    ax.axhline(0, color="grey", lw=0.8); ax.set_title(f"{h}-day IC by fold")
    ax.set_xlabel("fold")
axes[0].set_ylabel("rank IC"); axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

## Conclusion — the return ceiling on price data

Fill from the numbers above.

- **Does IC grow with horizon?** 5d ____ / 20d ____ / 60d ____ / 120d ____ — monotonic, or does it plateau?
- **Does it stay positive across folds** at the longer horizons, or get lumpier?
- **Annualised net spread** — larger at longer horizons after costs (fewer rebalances = less cost drag), or flat?
- **Ridge vs XGBoost** — still linear (Ridge >= XGB), or does structure appear at longer horizons?

### The decision on paid fundamentals

- **If IC keeps rising and stays consistent through 60–120 day** → a real, growing return
  signal lives at longer horizons on price data alone. That is the evidence that justifies
  buying point-in-time fundamentals (Sharadar): they are themselves monthly/quarterly
  signals, so they would *add to a horizon that already works*. Proceed to the paid step,
  test fundamentals at the best horizon here with the same A/B as short interest.
- **If IC plateaus or stays thin/lumpy through 120 day** → the return ceiling on price data
  is low, and lengthening does not rescue it. Paying for fundamentals becomes a weak bet:
  the horizon that would host them does not show reachable signal. The honest call is to
  ship the **volatility/risk engine** (validated, IC 0.48, beats GARCH) as the deliverable
  and keep returns as a thin cross-sectional tilt at the best free horizon found here.

Either way this is the last free lever: it prices the return opportunity before any money
is spent, exactly as the plan required.